# Processing GTFS Datasets

This script shows how I build the retrospective GTFS data and prepare the GTFS datasets for Actual vs Scheduled comparisons

There are FIVE objectives to be achieved from this notebook:
1) Processing Scheduled GTFS Datasets

2) Isolating Scheduled GTFS Datasets for Trams (only Manchester) 

3) Generating Retrospective GTFS Dataset

4) Additional Information about Retrospective Trips

5) Cleaning Rail GTFS Dataset

This code prepares the all the datasets for all the cities in one go.

---

The following datasets should be present in the 'Preprocessed' subfolder of the 'Data' directory before running this script:   

- Scheduled GTFS Datasets from [BODS Data Library](https://data.datalibrary.uk/transport/BODS-ARCHIVE/timetables/) (download only for North West, Yorkshire and South West for the selected dates)
- GTFS-RT Data from [BODS Data Library](https://data.datalibrary.uk/transport/BODS-ARCHIVE/gtfsrt/) (only keep the GTFS-RT data with a timestamp between 0600 - 0900 UTC, a limited data range can help to speed up processes)
- Rail Timetable from [Network Rail Data Portal](https://publicdatafeeds.networkrail.co.uk/ntrod/welcome) (see [Cleaning Rail GTFS Dataset](#4-cleaning-rail-gtfs-dataset) below on how to extract the data)

**Please install [`rt2gtfs`](https://github.com/kevinwinsper/rt2gtfs) before running the code. This new Python package is essential for Objective 3.**

In [ ]:
from pathlib import Path
import os
import pandas as pd
import geopandas as gpd
import zipfile
import io
from rt2gtfs import MatchingConfig, convert_rt_to_csv, construct_observed_gtfs
from shapely.geometry import LineString

ROOT = Path("../Data")
ROOT.resolve()

# setting links for easy reference later
pre_processed = ROOT/'Preprocessed'
post_processed = ROOT/'Postprocessed'
baseline_gtfs = ROOT/'Scenario 0_Baseline'
scheduled_gtfs = ROOT/'Scheduled'

# creating empty directories to store the outputs
os.makedirs(scheduled_gtfs, exist_ok=True)
os.makedirs(baseline_gtfs, exist_ok=True)

# Adjust target date and other dates accordingly
wednesdays = [
    '20260603', '20260610',
    '20260617', '20260624'
]

# dictionary of city and region
dict = {
    'Bristol': 'south_west',
    'Leeds': 'yorkshire',
    'Manchester': 'north_west'
}

# loading the study area with MSOA boundaries dissolved dataset 
# to be used for cropping scheduled GTFS
study_area = gpd.read_file(post_processed/'updated_studyarea.gpkg').to_crs(4326)

### 1) Processing Scheduled GTFS Datasets

This will scope down all TWELVE scheduled GTFS datasets so that it will only include trips where first and last stop is within the respective study areas.

In [ ]:
# Loop to run this for every city - Manchester, Nottingham and Bristol
for target_city, target_region in dict.items():

    for wed in wednesdays:

        studyarea_filtered = study_area[study_area['study_area'] == target_city]
    
        # For every date of the scheduled GTFS dataset, pull out the TXT files
        with zipfile.ZipFile(pre_processed/f'itm_{target_region}_gtfs_{wed}.zip', 'r') as z:
            agency = pd.read_csv(z.open('agency.txt'))
            calendar = pd.read_csv(z.open('calendar.txt'))
            calendar_dates = pd.read_csv(z.open('calendar_dates.txt'))
            feed_info = pd.read_csv(z.open('feed_info.txt'))
            routes = pd.read_csv(z.open('routes.txt'))
            shapes = pd.read_csv(z.open('shapes.txt'))
            stops = pd.read_csv(z.open('stops.txt'))
            trips = pd.read_csv(z.open('trips.txt'))
            stop_times = pd.read_csv(
                z.open('stop_times.txt'),
                usecols=[
                    'trip_id',
                    'arrival_time',
                    'departure_time',
                    'stop_id',
                    'stop_sequence'
                ]
            )

        stops_parent = stops.copy()

        # Filter for trips that serve at least one stop within the study area
        studyarea_stops = gpd.GeoDataFrame(
            stops,
            geometry=gpd.points_from_xy(
                stops['stop_lon'], 
                stops['stop_lat']
            ),
            crs='EPSG:4326'
        ).sjoin(
            studyarea_filtered,
            how='left',
            predicate='within'
        ).dropna(
            subset=['study_area']
        )['stop_id'].unique()

        studyarea_trips = stop_times.loc[
            stop_times['stop_id'].isin(studyarea_stops),
            'trip_id'
        ].unique()

        # Scope down the scheduled GTFS to have only the filtered trips
        # Cascade down every TXT file in the scheduled GTFS dataset
        stop_times = stop_times[stop_times['trip_id'].isin(studyarea_trips)].sort_values([
            'trip_id', 'stop_sequence'
        ])
        stops = stops[stops['stop_id'].isin(stop_times['stop_id'].unique())]
        if 'parent_station' in stops.columns:
            parent_ids = stops['parent_station'].dropna().unique()
            missing_parents = stops_parent[
                stops_parent['stop_id'].isin(parent_ids) &
                ~stops_parent['stop_id'].isin(stops['stop_id'])
            ]
            if not missing_parents.empty:
                stops = pd.concat([stops, missing_parents], ignore_index=True)
        trips = trips[trips['trip_id'].isin(studyarea_trips)]
        calendar = calendar[calendar['service_id'].isin(trips['service_id'].unique())]
        calendar_dates = calendar_dates[calendar_dates['service_id'].isin(trips['service_id'].unique())]
        shapes = shapes[shapes['shape_id'].isin(trips['shape_id'].unique())]
        routes = routes[routes['route_id'].isin(trips['route_id'].unique())]
        agency = agency[agency['agency_id'].isin(routes['agency_id'].unique())]
        feed_info['feed_end_date'] = 20260820

        # Save it into a new zipped GTFS dataset
        with zipfile.ZipFile(scheduled_gtfs/f'{target_city}_{wed}_sched.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zout:
            buffer = io.StringIO()
            stop_times.to_csv(buffer, index=False)
            zout.writestr('stop_times.txt', buffer.getvalue())

            buffer = io.StringIO()
            stops.to_csv(buffer, index=False)
            zout.writestr('stops.txt', buffer.getvalue())

            buffer = io.StringIO()
            trips.to_csv(buffer, index=False)
            zout.writestr('trips.txt', buffer.getvalue())

            buffer = io.StringIO()
            calendar.to_csv(buffer, index=False)
            zout.writestr('calendar.txt', buffer.getvalue())

            buffer = io.StringIO()
            calendar_dates.to_csv(buffer, index=False)
            zout.writestr('calendar_dates.txt', buffer.getvalue())

            buffer = io.StringIO()
            shapes.to_csv(buffer, index=False)
            zout.writestr('shapes.txt', buffer.getvalue())

            buffer = io.StringIO()
            routes.to_csv(buffer, index=False)
            zout.writestr('routes.txt', buffer.getvalue())

            buffer = io.StringIO()
            agency.to_csv(buffer, index=False)
            zout.writestr('agency.txt', buffer.getvalue())

            buffer = io.StringIO()
            feed_info.to_csv(buffer, index=False)
            zout.writestr('feed_info.txt', buffer.getvalue())

    print(f"{target_city}'s scheduled GTFS datasets have been scoped!")

### 2) Isolating Scheduled Tram GTFS Datasets (for Manchester only)

Scheduled GTFS downloaded from BODS Data Library includes tram services. However, there is no live location data on trams, so we don't have tram services in the retrospective GTFS datasets generated in Step 2.  

This meant that modelling actual travelling patterns only models actual BUS performance, but trams remain based on scheduled performance. This is a known limitation and beyond the focus of my MSc dissertation. I just need to have a separate GTFS for trams in Manchester for me to feed into r5py when modelling actual travelling patterns, either at baseline or any of the improvement scenarios.

In [ ]:
for wed in wednesdays:

    # extract the scoped scheduled GTFS data
    with zipfile.ZipFile(scheduled_gtfs/f'Manchester_{wed}_sched.zip', 'r') as z:
        agency = pd.read_csv(z.open('agency.txt'))
        calendar = pd.read_csv(z.open('calendar.txt'))
        calendar_dates = pd.read_csv(z.open('calendar_dates.txt'))
        feed_info = pd.read_csv(z.open('feed_info.txt'))
        routes = pd.read_csv(z.open('routes.txt'))
        shapes = pd.read_csv(z.open('shapes.txt'))
        stops = pd.read_csv(z.open('stops.txt'))
        trips = pd.read_csv(z.open('trips.txt'))
        stop_times = pd.read_csv(
            z.open('stop_times.txt'),
            usecols=[
                'trip_id',
                'arrival_time',
                'departure_time',
                'stop_id',
                'stop_sequence'
            ]
        )

    # tram services have route_type == 0
    # cascade down every TXT file in the scheduled GTFS dataset
    routes = routes[routes['route_type'] == 0]
    agency = agency[agency['agency_id'].isin(routes['agency_id'].unique())]
    trips = trips[trips['route_id'].isin(routes['route_id'].unique())]
    calendar = calendar[calendar['service_id'].isin(trips['service_id'].unique())]
    calendar_dates = calendar_dates[calendar_dates['service_id'].isin(trips['service_id'].unique())]
    stop_times = stop_times[stop_times['trip_id'].isin(trips['trip_id'].unique())]
    stops = stops[stops['stop_id'].isin(stop_times['stop_id'].unique())]
    shapes = shapes[shapes['shape_id'].isin(trips['shape_id'].unique())]

    # Save it into a new zipped GTFS dataset
    with zipfile.ZipFile(scheduled_gtfs/f'Manchester_{wed}_tram.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zout:
        buffer = io.StringIO()
        stop_times.to_csv(buffer, index=False)
        zout.writestr('stop_times.txt', buffer.getvalue())

        buffer = io.StringIO()
        stops.to_csv(buffer, index=False)
        zout.writestr('stops.txt', buffer.getvalue())

        buffer = io.StringIO()
        trips.to_csv(buffer, index=False)
        zout.writestr('trips.txt', buffer.getvalue())

        buffer = io.StringIO()
        calendar.to_csv(buffer, index=False)
        zout.writestr('calendar.txt', buffer.getvalue())

        buffer = io.StringIO()
        calendar_dates.to_csv(buffer, index=False)
        zout.writestr('calendar_dates.txt', buffer.getvalue())

        if len(shapes) != 0:
            buffer = io.StringIO()
            shapes.to_csv(buffer, index=False)
            zout.writestr('shapes.txt', buffer.getvalue())

        buffer = io.StringIO()
        routes.to_csv(buffer, index=False)
        zout.writestr('routes.txt', buffer.getvalue())

        buffer = io.StringIO()
        agency.to_csv(buffer, index=False)
        zout.writestr('agency.txt', buffer.getvalue())

        buffer = io.StringIO()
        feed_info.to_csv(buffer, index=False)
        zout.writestr('feed_info.txt', buffer.getvalue())

print("Manchester's tram GTFS datasets are scoped!")

### 3) Generating Retrospective GTFS Data

Based on the `rt2gtfs` package, this will be a two-step process where all the binary files must first be converted into a CSV file, and then building up the data into a valid GTFS dataset. 

In [ ]:
# Step 1: Converting GTFS-RT data into one CSV file
# This can be run once because the GTFS-RT data covers the whole country
# So the same CSV data will be used thrice in Step 2

for wed in wednesdays:

    csv_config = MatchingConfig(
        gtfs_dir=scheduled_gtfs,
        rt_dir=pre_processed,
        rt_foldername_template=f'gtfsrt_{wed}',
        result_dir=baseline_gtfs,
        unzip_rt=True,
        deduplicate_rt=True
    )

    convert_rt_to_csv(wed, csv_config)

In [ ]:
# Step 2: Building retrospective GTFS data
for target_city in dict:

    for wed in wednesdays:

        gtfs_config = MatchingConfig(
            gtfs_dir=scheduled_gtfs,
            gtfs_filename_template=f'{target_city}_{{date}}_sched',
            rt_dir=pre_processed/'csv',
            result_dir=baseline_gtfs,
            timetable_region=target_city,
            check_extra_dates=True,
            extra_dates=[d for d in wednesdays if d != wed],
            matching_periods=[("07:00:00", "10:00:00")]
        )

        construct_observed_gtfs(wed, gtfs_config)

    print(f"{target_city}'s retrospective GTFS dataset is generated!")

`rt2gtfs` does not include details about routes or agency info when recreating GTFS from real-time data. This needs to be brought in so that the `r5py` modelling in Script 4 could be operationalised

In [ ]:
for target_city in dict:

    for wed in wednesdays:

        retro_dir = baseline_gtfs/target_city/wed
        sched_dir = scheduled_gtfs/f'{target_city}_{wed}_sched'

        sched_agency = pd.read_csv(sched_dir/'agency.txt')
        sched_routes = pd.read_csv(sched_dir/'routes.txt')
        retro_trips = pd.read_csv(retro_dir/'trips.txt')
        retro_feedinfo = pd.read_csv(retro_dir/'feed_info.txt')

        retro_routes = sched_routes[sched_routes['route_id'].isin(retro_trips['route_id'].unique())]
        retro_routes.to_csv(retro_dir/'routes.txt', index=False)

        retro_agency = sched_agency[sched_agency['agency_id'].isin(retro_routes['agency_id'].unique())]
        retro_agency.to_csv(retro_dir/'agency.txt', index=False)

        retro_feedinfo = retro_feedinfo.iloc[[0]]
        retro_feedinfo.to_csv(retro_dir/'feed_info.txt', index=False)

        # zip everything in retro_dir, flat (no subfolder inside the zip)
        zip_path = baseline_gtfs/target_city/f'{target_city}_{wed}.zip'  # or wherever you want it saved
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in retro_dir.glob('*.txt'):
                zf.write(f, arcname=f.name)

### 4) Additional Information on Retrospective Trip Patterns

I want to have more information from the scheduled scoped GTFS, which have been unzipped from the `rt2gtfs` conversion phase, to facilitate scenario generation in Script 3.

This information should be a table of trip patterns (with routes and agency names) that happen in the morning peak hour (0700 - 1000), with a list of trip_ids that run that pattern and a count of items in that list (frequency of that pattern in the morning), with patterns that run less than six times filtered out. That table should also have a marker if the pattern is city-centre inbound, and if they also serve most-deprived MSOAs.

In [ ]:
# List of functions

def to_sec(t):

    """Returns seconds after midnight for a given HH:MM:SS string in 
    departure_time."""
    
    h, m, s = t.strip().split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


# Morning peak hour window for filtering
WINDOW_START = to_sec("07:00:00")
WINDOW_END = to_sec("10:00:00")


def inmorning_trips(stop_times):

    """Returns a set of trips that have their first stop within the morning peak 
    hour window."""

    idx = stop_times.groupby('trip_id')['stop_sequence'].idxmin()
    first_stop = stop_times.loc[idx]

    intime = first_stop[
        (first_stop['departure_time'].apply(to_sec) >= WINDOW_START) &
        (first_stop['departure_time'].apply(to_sec) <= WINDOW_END)
    ]
    
    return set(intime['trip_id'])


def incentre_stops(stops, scoped_cc):

    """Returns a set of stops that are within the city centre."""

    stops_gdf = gpd.GeoDataFrame(
        stops,
        geometry=gpd.points_from_xy(
            stops['stop_lon'], 
            stops['stop_lat']
        ),
        crs='EPSG:4326'
    )

    centre_stops = stops_gdf.sjoin(
        scoped_cc,
        how='left',
        predicate='intersects'
    ).dropna(
        subset=['pua']
    )['stop_id'].unique()
    
    return set(centre_stops)


def indeprived_stops(stops, scoped_deprived):

    """Returns a set of stops that are within most deprived MSOAs."""

    stops_gdf = gpd.GeoDataFrame(
        stops,
        geometry=gpd.points_from_xy(
            stops['stop_lon'], 
            stops['stop_lat']
        ),
        crs='EPSG:4326'
    )

    deprived_stops = stops_gdf.sjoin(
        scoped_deprived,
        how='left',
        predicate='intersects'
    ).dropna(
        subset=['study_area']
    )['stop_id'].unique()
    
    return set(deprived_stops)


def trip_pattern(stop_times, trip_ids):

    """Returns a DataFrame of every morning trip with their corresponding trip 
    pattern (stops in order from start to finish) as tuples."""

    filtered_stoptimes = stop_times[stop_times['trip_id'].isin(trip_ids)]

    filtered_wpattern = filtered_stoptimes.groupby(
        'trip_id'
    )['stop_id'].apply(
        tuple
    ).rename(
        'pattern'
    ).reset_index()

    return (filtered_wpattern)


def classify_inbound(pattern, centre_stops):

    """Classifies a trip pattern if it is an inbound city-centre pattern or not.
    Inbound city-centre patterns are defined as patterns where its second half 
    contains at least one stop within the city centre."""

    n = len(pattern)

    half = n // 2

    in_centre = [stop_id in centre_stops for stop_id in pattern]

    if any(in_centre[half:]):
        return 'inbound'
    else:
        return 'not_inbound'


def classify_deprived(pattern, deprived_stops):

    """Identifies when a trip pattern calls at stop(s) within most deprived MSOAs."""

    in_deprived = [stop_id in deprived_stops for stop_id in pattern]

    if any(in_deprived):
        return 'deprived'
    else:
        return 'not_indeprived'
    

def classify_frequency(frequency):

    """Categorises frequency of trip patterns"""

    if frequency < 5:
        return 'too_infrequent'
    elif 5 <= frequency < 12:
        return 'not_frequent'
    else:
        return 'frequent'


# Loading GPKG files meant for GTFS scoping
city_centre = gpd.read_file(
    post_processed/'updated_city_centre.gpkg'
).to_crs(4326)

deprived = gpd.read_file(
    post_processed/'deprived_msoa.gpkg'
).to_crs(4326)

file_names = ['agency', 'routes', 'trips', 'stop_times', 'stops']

In [ ]:
for target_city in dict:

    # -----
    # STEP 1: Aggregate info from unzipped scheduled GTFS across all four Wednesdays.
    combined = {name: [] for name in file_names}

    for wed in wednesdays:
        city_date_dir = baseline_gtfs/target_city/wed

        for name in file_names:
            df = pd.read_csv(city_date_dir/f'{name}.txt')
            if name == 'trips':
                df['source_wed'] = wed
            combined[name].append(df)

    # deduplicating similar records over multiple days
    agency = pd.concat(combined['agency'], ignore_index=True).drop_duplicates()
    routes = pd.concat(combined['routes'], ignore_index=True).drop_duplicates()
    stops = pd.concat(combined['stops'], ignore_index=True).drop_duplicates(subset='stop_id', keep='last')

    # ensures trip_ids in each day is unique
    trips = pd.concat(combined['trips'], ignore_index=True).drop_duplicates()

    # ensures that trip_ids that are shared over multiple days have the same pattern (same stop sequence)
    stop_times = pd.concat(
        combined['stop_times'], 
        ignore_index=True
    ).sort_values(
        ['trip_id', 'stop_sequence']
    ).drop_duplicates(
        subset=['trip_id', 'stop_sequence'], 
        keep='last'
    )

    # -----
    # STEP 2: Create TWO Lookup tables

    # Lookup 1 - trip_id -> route_id: one row per trip_id to consolidate a full list of trips associated with a route branding
    route_lookup = trips[['trip_id', 'route_id']].drop_duplicates(subset='trip_id')

    # Lookup 2 - trip_id -> source_wed: just a smaller dataframe with cols for trip_id and the day in which the service ran
    trip_date_lookup = trips[['trip_id', 'source_wed']].drop_duplicates()

    # -----
    # STEP 3: Scope down Lookup 1 for trips that STARTED during morning peak hours
    morning_tripids = inmorning_trips(stop_times)

    morningtrips_wpattern = trip_pattern(
        stop_times, 
        morning_tripids
    ).merge(
        route_lookup,
        on='trip_id',
        how='left'
    )

    # -----
    # STEP 4: Convert scoped-down Lookup 1 data into intermediate DF 1: info on all the stops in a sequence, and trip_ids which follow that sequence
    morning_patterns = morningtrips_wpattern.sort_values(
        ['route_id', 'pattern']
    ).groupby(
        ['route_id', 'pattern']
    )['trip_id'].apply(
        list
    ).reset_index().rename(
        columns={'trip_id': 'id_list'}
    )

    # -----
    # STEP 5: Merge scoped-down Lookup 1 with Lookup 2 to create intermediate DF 2: info omn median number of trips per each pattern over the four days
    pattern_date_counts = morningtrips_wpattern.merge(
        trip_date_lookup, 
        on='trip_id', 
        how='left'
    ).groupby(
        [
            'route_id', 
            'pattern', 
            'source_wed'
        ]
    ).size().unstack(
        'source_wed', 
        fill_value=0
    )

    for wed in wednesdays:
        if wed not in pattern_date_counts.columns:
            pattern_date_counts[wed] = 0

    pattern_date_counts['median_trips'] = pattern_date_counts.median(axis=1)
    pattern_date_counts = pattern_date_counts.reset_index()

    # -----
    # STEP 6: Merge intermediate DFs 1 and 2
    # Then identify pattern frequency
    morningpat_wcounts = morning_patterns.merge(
        pattern_date_counts[['route_id', 'pattern', 'median_trips']], 
        on=['route_id', 'pattern'], 
        how='left'
    )

    morningpat_wcounts['frequent?'] = morningpat_wcounts['median_trips'].apply(classify_frequency)

    # -----
    # STEP 7: Preparing the filters
    # Firstly, Scope down the GPKG filters
    scoped_cc = city_centre[city_centre['pua'] == target_city]
    scoped_deprived = deprived[deprived['study_area'] == target_city]

    # Identify stops in city centre or in deprived MSOAs
    centre_stops = incentre_stops(stops, scoped_cc)
    deprived_stops = indeprived_stops(stops, scoped_deprived)

    # -----
    # STEP 7: Find out if the trip patterns in the merged intermediate DFs are inbound or not, 
    # and if they serve high demand/most deprived MSOAs
    morningpat_wcounts['inbound?'] = morningpat_wcounts['pattern'].apply(
        lambda x: classify_inbound(x, centre_stops)
    )
    morningpat_wcounts['deprived?'] = morningpat_wcounts['pattern'].apply(
        lambda x: classify_deprived(x, deprived_stops)
    )

    # -----
    # STEP 8: Add info about the route branding associated with each pattern and the agency that operates it
    route_info = routes[['route_id', 'route_short_name', 'agency_id']].drop_duplicates().merge(
        agency[['agency_id', 'agency_name']],
        on='agency_id',
        how='left'
    )

    full_morning_patterns = route_info[['route_id', 'route_short_name', 'agency_name']].merge(
        morningpat_wcounts, 
        on='route_id', 
        how='right'
    )

    full_morning_patterns.to_csv(post_processed/f'{target_city}_retro_morning_patterns.csv', index=False)

    print(f"{target_city}'s retrospective morning trip patterns are generated!")

### 5) Cleaning Rail GTFS Dataset

Travel time modelling for any of the scenarios (baseline vs any improvement scenario) requires the inclusion of rail GTFS as well. Frustratingly, Network Rail **DOES NOT** provide a GTFS dataset. Instead, rail timetables are provided in a CIF format that needs to be converted, and this conversion needs to be done on R. So the following are the steps I did to get my hands on the rail schedules in GTFS format

1) Sign up for an account at the [Network Rail Data Portal](https://publicdatafeeds.networkrail.co.uk/ntrod/welcome).
2) Follow steps to  to [pull the timetable data](https://wiki.openraildata.com/index.php?title=SCHEDULE) into the 'Preprocess' subfolder of the 'Data' directory.
3) Use [`uk2gtfs`](https://itsleeds.github.io/UK2GTFS/) R library to convert the CIF files into GTFS. I used the following code:  

```
library(UK2GTFS)
gtfs_write(
  nr2gtfs(
    path_in = {path to the zipped CIF rail timetable data in 'Preprocessed' subfolder of 'Data' directory},
    ncores  = 16,
    silent  = FALSE
  ), 
  folder = {path to 'Preprocessed' subfolder of 'Data' directory}, 
  name = "rail_gtfs")

``` 

What you will notice if you validate this transformed rail GTFS is that it has several issues: 
1) missing route_types in routes.txt. Fill them up with route_type == 2 (indicating heavy rail)
2) agency_id in routes.txt does not exist in agency.txt because these are for freight. Filter out freight routes.
3) some stops in stop_times.txt do not exist with stops.txt. Filter out entire trips that call at these phantom stops.

In [ ]:
# pull out the TXT files from rail GTFS data
with zipfile.ZipFile(pre_processed/f'rail_gtfs.zip', 'r') as z:
    agency = pd.read_csv(z.open('agency.txt'))
    calendar = pd.read_csv(z.open('calendar.txt'))
    calendar_dates = pd.read_csv(z.open('calendar_dates.txt'))
    routes = pd.read_csv(z.open('routes.txt'))
    stops = pd.read_csv(z.open('stops.txt'))
    trips = pd.read_csv(z.open('trips.txt'))
    stop_times = pd.read_csv(
        z.open('stop_times.txt'),
        usecols=[
            'trip_id',
            'arrival_time',
            'departure_time',
            'stop_id',
            'stop_sequence'
        ]
    )

# Fix 1: fill missing route_type (heavy rail = 2)
routes['route_type'] = routes['route_type'].fillna(2).astype(int)

# Fix 2: filter out freight routes
# cascade down every TXT file
routes = routes[routes['agency_id'].isin(agency['agency_id'].unique())]
trips = trips[trips['route_id'].isin(routes['route_id'].unique())]
calendar = calendar[calendar['service_id'].isin(trips['service_id'].unique())]
calendar_dates = calendar_dates[calendar_dates['service_id'].isin(trips['service_id'].unique())]
stop_times = stop_times[stop_times['trip_id'].isin(trips['trip_id'].unique())]
stops = stops[stops['stop_id'].isin(stop_times['stop_id'].unique())]

# Fix 3: filter out trips with phantom stops
# re-cascade it down every TXT file
phantom_trips = stop_times.loc[
    ~stop_times['stop_id'].isin(stops['stop_id'].unique()),
    'trip_id'
].unique()

stop_times = stop_times[~stop_times['trip_id'].isin(phantom_trips)].sort_values([
    'trip_id', 'stop_sequence'
])
stops = stops[stops['stop_id'].isin(stop_times['stop_id'].unique())]
trips = trips[~trips['trip_id'].isin(phantom_trips)]
calendar = calendar[calendar['service_id'].isin(trips['service_id'].unique())]
calendar_dates = calendar_dates[calendar_dates['service_id'].isin(trips['service_id'].unique())]
routes = routes[routes['route_id'].isin(trips['route_id'].unique())]
agency = agency[agency['agency_id'].isin(routes['agency_id'].unique())]

# Save it into a new zipped GTFS dataset
with zipfile.ZipFile(scheduled_gtfs/'railgtfs_cleaned.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zout:
    buffer = io.StringIO()
    stop_times.to_csv(buffer, index=False)
    zout.writestr('stop_times.txt', buffer.getvalue())

    buffer = io.StringIO()
    stops.to_csv(buffer, index=False)
    zout.writestr('stops.txt', buffer.getvalue())

    buffer = io.StringIO()
    trips.to_csv(buffer, index=False)
    zout.writestr('trips.txt', buffer.getvalue())

    buffer = io.StringIO()
    calendar.to_csv(buffer, index=False)
    zout.writestr('calendar.txt', buffer.getvalue())

    buffer = io.StringIO()
    calendar_dates.to_csv(buffer, index=False)
    zout.writestr('calendar_dates.txt', buffer.getvalue())

    buffer = io.StringIO()
    routes.to_csv(buffer, index=False)
    zout.writestr('routes.txt', buffer.getvalue())

    buffer = io.StringIO()
    agency.to_csv(buffer, index=False)
    zout.writestr('agency.txt', buffer.getvalue())

print('Rail GTFS has been cleaned')